# Creating a Mesh for the ChipShop 560 Device

Here we are creating a mesh for the Chip 560 Device using gmsh.

We gathered the dimensions from the spec sheet below:


<img src="/home/dcg33/ChipShop_Zonated_Chip/data/Device560.PNG" width=500>

Those dimensions are list in the table below:

| Dimension | Value (mm) |
|--|--|
|Channel Length (L) | 58.5 |
|Channel Width (W) | 2.5 |
|Channel Height (H) | 0.35 |

In [5]:
import pyvista as pv
import meshio as mio
import plotly.express as px
import gmsh

from trame_pyvista.jupyter import launch_server
await launch_server().ready

gmsh.initialize()
data_dir = "../data/"

In [6]:
gmsh.model.add("channel")

length = 58.5 #mm
width = 2.5 #mm
height = 0.35 #mm

box = gmsh.model.occ.addBox(
    0, 0, 0,       # x, y, z position
    length,
    width,
    height
)


gmsh.model.occ.synchronize()

gmsh.model.mesh.generate(3)
gmsh.write(f"{data_dir}device560_init.msh")

gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 5 (Line)
Info    : [ 50%] Meshing curve 6 (Line)
Info    : [ 60%] Meshing curve 7 (Line)
Info    : [ 60%] Meshing curve 8 (Line)
Info    : [ 70%] Meshing curve 9 (Line)
Info    : [ 80%] Meshing curve 10 (Line)
Info    : [ 90%] Meshing curve 11 (Line)
Info    : [100%] Meshing curve 12 (Line)
Info    : Done meshing 1D (Wall 0.00113258s, CPU 0.001961s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 2 (Plane, Frontal-Delaunay)
Info    : [ 40%] Meshing surface 3 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 4 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 5 (Plane, Frontal-Delaunay)
Info    : [ 90%] Meshing surface 6 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.00202329s

In [7]:
mesh = mio.read(f"{data_dir}device560_init.msh")

# Keep only the 3D volume elements
tetra = mesh.cells_dict["tetra"]

mesh_3d = mio.Mesh(
    points=mesh.points,
    cells={"tetra": tetra},
)

pv_mesh = pv.from_meshio(mesh_3d)

pv_mesh.plot(show_edges=True)

Widget(value='<iframe src="http://localhost:41471/index.html?ui=P_0x70acfe28c050_2&reconnect=auto" class="pyvi…

In [8]:
import gmsh
import sys

# 1. Initialize Gmsh
gmsh.initialize()
gmsh.model.add("StraightChannel_Layered")

# --- Dimensions (Converted to meters or mm; using mm here for clarity) ---
L = 58.5       # Channel Length (mm)[cite: 1]
W = 2.5        # Channel Width (mm)[cite: 1]
H_tot = 0.35   # Total Channel Depth (mm)[cite: 1]

H_gel = 0.10   # Hydrogel Matrix Layer Height (bottom)
H_fluid = H_tot - H_gel  # Free Fluid Layer Height (top)

# Enable OpenCASCADE geometry kernel
occ = gmsh.model.occ

# 2. Create Volumetric Domains (Boxes)
# Layer 1: Hydrogel Matrix Layer (Bottom domain: z from 0 to H_gel)
tag_gel = occ.addBox(0, 0, 0, L, W, H_gel)

# Layer 2: Free Fluid Domain (Top domain: z from H_gel to H_tot)
tag_fluid = occ.addBox(0, 0, H_gel, L, W, H_fluid)

# Synchronize CAD representation with Gmsh internal representation
occ.synchronize()

# 3. Form Conformal Boundaries Interface
# Fragment ensures shared faces between touching volumes so nodes align in mesh
out_dim_tags, out_dim_tags_map = occ.fragment([(3, tag_gel)], [(3, tag_fluid)])
occ.synchronize()

# 4. Define Physical Groups for Subdomains and Boundaries
# Physical Volumes (Subdomains)
# Note: fragment preserves original volume IDs or updates mapping
gmsh.model.addPhysicalGroup(3, [tag_gel], tag=1, name="HydrogelDomain")
gmsh.model.addPhysicalGroup(3, [tag_fluid], tag=2, name="FluidDomain")

# Retrieve surface entities to set boundary conditions
surfaces = gmsh.model.getEntities(dim=2)

cell_surface = []
inlet_surfaces = []
outlet_surfaces = []

for surf in surfaces:
    tag = surf[1]
    # Get center of mass of the surface to identify specific boundaries
    mass_center = gmsh.model.occ.getCenterOfMass(2, tag)
    x, y, z = mass_center[0], mass_center[1], mass_center[2]
    
    # Bottom Surface (Cell Layer at z = 0)
    if abs(z - 0.0) < 1e-4:
        cell_surface.append(tag)
    # Inlet Surfaces (x = 0)
    elif abs(x - 0.0) < 1e-4:
        inlet_surfaces.append(tag)
    # Outlet Surfaces (x = L)
    elif abs(x - L) < 1e-4:
        outlet_surfaces.append(tag)

# Assign Boundary Physical Surfaces
gmsh.model.addPhysicalGroup(2, cell_surface, tag=101, name="CellLayer")
gmsh.model.addPhysicalGroup(2, inlet_surfaces, tag=102, name="Inlet")
gmsh.model.addPhysicalGroup(2, outlet_surfaces, tag=103, name="Outlet")

# 5. Control Mesh Refinement
# Set fine element size near cell layer / hydrogel interface
gmsh.option.setNumber("Mesh.MeshSizeMin", 0.02)  # 20 um
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.20)  # 200 um

# Generate 3D Tetrahedral Mesh
gmsh.model.mesh.generate(3)

# Save mesh for solver
gmsh.write(f"{data_dir}layered_channel.msh")

# Launch Gmsh GUI to visually inspect layers (optional)
if "-nobase" not in sys.argv:
    gmsh.fltk.run()

gmsh.finalize()

Info    : Meshing 1D...                                                                                                     
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 20%] Meshing curve 4 (Line)
Info    : [ 30%] Meshing curve 5 (Line)
Info    : [ 30%] Meshing curve 6 (Line)
Info    : [ 40%] Meshing curve 7 (Line)
Info    : [ 40%] Meshing curve 8 (Line)
Info    : [ 50%] Meshing curve 9 (Line)
Info    : [ 50%] Meshing curve 10 (Line)
Info    : [ 60%] Meshing curve 11 (Line)
Info    : [ 60%] Meshing curve 12 (Line)
Info    : [ 70%] Meshing curve 13 (Line)
Info    : [ 70%] Meshing curve 14 (Line)
Info    : [ 80%] Meshing curve 15 (Line)
Info    : [ 80%] Meshing curve 16 (Line)
Info    : [ 90%] Meshing curve 17 (Line)
Info    : [ 90%] Meshing curve 18 (Line)
Info    : [100%] Meshing curve 19 (Line)
Info    : [100%] Meshing curve 20 (Line)
Info    : Done meshing 1D (Wall 0.00200659s, CPU 0.002811s)
Info

In [9]:
mesh = mio.read(f"{data_dir}layered_channel.msh")

# Keep only the 3D volume elements
tetra = mesh.cells_dict["tetra"]

mesh_3d = mio.Mesh(
    points=mesh.points,
    cells={"tetra": tetra},
)

pv_mesh = pv.from_meshio(mesh_3d)

pv_mesh.plot(show_edges=True)

Widget(value='<iframe src="http://localhost:41471/index.html?ui=P_0x70ace885f8f0_3&reconnect=auto" class="pyvi…